<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%89%B5%E5%BB%BA%E8%B3%87%E6%96%99%E9%9B%86/%E9%87%8D%E6%96%B0%E5%88%87%E5%89%B2%E8%B3%87%E6%96%99%E9%9B%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

# --- 📁 Step 0: 掛載雲端硬碟與讀取資料 ---
drive.mount('/content/drive')

print("📥 正在載入所有資料集...")
# 1. 讀取「完美交集」的主資料集 (約 10 萬筆，包含完整的 target)
df_intersection = pd.read_csv("/content/drive/MyDrive/畢業專題資料集/phishing_dataset_COMPLETE.csv")

# 2. 讀取各自的原始特徵資料集 (包含尚未對齊的原始龐大資料)
df_url_raw = pd.read_csv("/content/drive/MyDrive/畢業專題資料集/phishing_dataset_URL.csv")
df_html_raw = pd.read_csv("/content/drive/MyDrive/畢業專題資料集/phishing_dataset_HTML.csv")
df_html_raw = df_html_raw[df_html_raw['feature_extracted'] == 1]
df_ai_raw = pd.read_csv("/content/drive/MyDrive/畢業專題資料集/phishing_dataset_AI.csv")
df_ai_raw = df_ai_raw[df_ai_raw['ai_status'] == 'AI_SUCCESS']

# 確保所有 URL 都是乾淨的字串格式，避免比對失敗
for df in [df_intersection, df_url_raw, df_html_raw, df_ai_raw]:
    df['url'] = df['url'].astype(str).str.strip()

# --- 🔪 Step 1: 按照比例拆分元模型資料集 (Stratified Split) ---
print("\n⚖️ 執行分層抽樣拆分 (維持 Target 比例)...")

# 從交集資料中，切出 60% 作為 Meta Model 專屬的訓練/測試資料
df_base_pool, df_meta_set = train_test_split(
    df_intersection,
    test_size=0.50, # 50% 給元模型
    stratify=df_intersection['target'],
    random_state=42 # 設定亂數種子，確保每次重跑結果都一樣
)

print(f"✅ Meta_Set (元模型專屬): {len(df_meta_set)} 筆")
print(f"✅ Base_Pool (交集剩餘): {len(df_base_pool)} 筆")

# 驗證比例是否完美繼承
print("\n📊 驗證 Meta_Set 中的 Target 比例:")
print(df_meta_set['target'].value_counts(normalize=True).round(4) * 100)

# --- 🛡️ Step 2: 從各自特徵集中「徹底排除」 Meta_Set ---
print("\n從基模型資料集中剔除 Meta_Set 網址...")

# 將 Meta_Set 的 url 轉成 Python 的 Set (集合)，這會讓比對速度快上 100 倍
meta_urls_forbidden = set(df_meta_set['url'].tolist())

# 使用 Pandas 的 .isin() 搭配 ~ (反向選擇) 來進行過濾
df_url_train = df_url_raw[~df_url_raw['url'].isin(meta_urls_forbidden)]
df_html_train = df_html_raw[~df_html_raw['url'].isin(meta_urls_forbidden)]
df_ai_train = df_ai_raw[~df_ai_raw['url'].isin(meta_urls_forbidden)]

print(f"✅ URL 基模型訓練集: 從 {len(df_url_raw)} 筆 ➡️ 縮減為 {len(df_url_train)} 筆")
print(f"✅ HTML 基模型訓練集: 從 {len(df_html_raw)} 筆 ➡️ 縮減為 {len(df_html_train)} 筆")
print(f"✅ AI 基模型訓練集: 從 {len(df_ai_raw)} 筆 ➡️ 縮減為 {len(df_ai_train)} 筆")

print("\n📊 驗證 基模型訓練集 中的 Target 比例:")
print('===============URL============')
print(df_url_train['target'].value_counts(normalize=True).round(4) * 100)
print('===============HTML============')
print(df_html_train['target'].value_counts(normalize=True).round(4) * 100)
print('===============AI============')
print(df_ai_train['target'].value_counts(normalize=True).round(4) * 100)

# --- 儲存資料集 ---
print("\n正在儲存切分好的訓練集...")

# 儲存元模型專屬資料 (這份未來用來評估基模型、訓練 Meta Model)
df_meta_set.to_csv("/content/drive/MyDrive/畢業專題資料集/Dataset_Meta_Set.csv", index=False)

# 儲存基模型各自的專屬訓練集 (他們已經絕對看不到 Meta_Set 的東西了)
df_url_train.to_csv("/content/drive/MyDrive/畢業專題資料集/Train_Base_URL.csv", index=False)
df_html_train.to_csv("/content/drive/MyDrive/畢業專題資料集/Train_Base_HTML.csv", index=False)
df_ai_train.to_csv("/content/drive/MyDrive/畢業專題資料集/Train_Base_AI.csv", index=False)

print("🎉 資料集切割與過濾徹底完成！沒有任何 Data Leakage！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 正在載入所有資料集...

⚖️ 執行分層抽樣拆分 (維持 Target 比例)...
✅ Meta_Set (元模型專屬): 54590 筆
✅ Base_Pool (交集剩餘): 54589 筆

📊 驗證 Meta_Set 中的 Target 比例:
target
1    88.12
0    11.88
Name: proportion, dtype: float64

從基模型資料集中剔除 Meta_Set 網址...
✅ URL 基模型訓練集: 從 247225 筆 ➡️ 縮減為 192611 筆
✅ HTML 基模型訓練集: 從 125618 筆 ➡️ 縮減為 71005 筆
✅ AI 基模型訓練集: 從 135646 筆 ➡️ 縮減為 81036 筆

📊 驗證 基模型訓練集 中的 Target 比例:
===============URL============
target
0    52.0
1    48.0
Name: proportion, dtype: float64
===============HTML============
target
1    82.42
0    17.58
Name: proportion, dtype: float64
===============AI============
target
1    81.61
0    18.39
Name: proportion, dtype: float64

正在儲存切分好的訓練集...
🎉 資料集切割與過濾徹底完成！沒有任何 Data Leakage！
